# Project Lab 2: Detection, Mini-GPT and a Sequence Model

Notebook for **Chapter 16** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch16.html

Tip: change a number, predict the output, then run the cell again.

## 16.1 Project: Object Detection on a Custom Dataset (learner project #36)

In [ ]:
import numpy as np

def coco_to_xyxy(b):  x, y, w, h = b; return [x, y, x + w, y + h]
def xyxy_to_yolo(b, W, H):
    x1, y1, x2, y2 = b
    return [((x1 + x2) / 2) / W, ((y1 + y2) / 2) / H, (x2 - x1) / W, (y2 - y1) / H]
def yolo_to_xyxy(b, W, H):
    cx, cy, w, h = b
    return [(cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H]

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1]); ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0.0

def nms(boxes, scores, thr=0.5):
    order = [int(i) for i in np.argsort(scores)[::-1]]; keep = []
    while order:
        i = order.pop(0); keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) < thr]
    return keep

def average_precision(preds, gts, thr=0.5):
    """preds: list of (image_id, score, box); gts: dict image_id -> list of boxes"""
    npos = sum(len(v) for v in gts.values()); used = {k: [False]*len(v) for k, v in gts.items()}
    preds = sorted(preds, key=lambda p: -p[1]); tp = []
    for img, score, box in preds:
        best, bj = 0, -1
        for j, g in enumerate(gts.get(img, [])):
            o = iou(box, g)
            if o > best: best, bj = o, j
        if best >= thr and not used[img][bj]: used[img][bj] = True; tp.append(1)
        else: tp.append(0)
    tp = np.array(tp); ctp = np.cumsum(tp); cfp = np.cumsum(1 - tp)
    rec = ctp / npos; prec = ctp / (ctp + cfp)
    mrec = np.concatenate([[0], rec, [1]]); mpre = np.concatenate([[0], prec, [0]])
    for i in range(len(mpre) - 2, -1, -1): mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]).sum()), rec, prec

In [ ]:
c = [50, 40, 100, 60]; x = coco_to_xyxy(c); print("coco->xyxy", x)
y = xyxy_to_yolo(x, 640, 480); print("yolo", [round(v, 4) for v in y])
print("round trip", yolo_to_xyxy(y, 640, 480))
print("IoU same", iou(x, x), " IoU shifted", round(iou([0,0,10,10],[5,0,15,10]), 4), " IoU apart", iou([0,0,1,1],[5,5,6,6]))
boxes = [[10,10,50,50],[12,12,52,52],[100,100,140,140],[11,9,49,51]]; sc = [0.9, 0.8, 0.7, 0.6]
print("NMS keep", nms(boxes, sc))
gts = {"a": [[10,10,50,50],[100,100,140,140]], "b": [[20,20,60,60]]}
preds = [("a",0.95,[11,11,50,50]),("a",0.9,[12,12,51,51]),("b",0.8,[70,70,110,110]),
         ("a",0.7,[100,100,141,140]),("b",0.6,[21,19,60,61])]
ap, rec, prec = average_precision(preds, gts)
print("recall", np.round(rec, 3)); print("precision", np.round(prec, 3)); print("AP@0.5", round(ap, 4))
print("AP@0.95", round(average_precision(preds, gts, 0.95)[0], 4))

In [ ]:
import torch, torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_f = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_f, num_classes=3)   # background + 2 classes
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=5e-4)
# each batch: images = list of 3xHxW float tensors in [0,1]
#             targets = list of {"boxes": FloatTensor[N,4] (xyxy pixels), "labels": Int64Tensor[N] (1..K)}
# model.train(); loss_dict = model(images, targets); sum(loss_dict.values()).backward(); opt.step()
# model.eval(); outputs = model(images)  -> boxes, labels, scores, then NMS is already applied

## 16.2 Project: Mini-GPT From Scratch (learner project #37)

In [ ]:
text = ("the shop opens at nine. the shop closes at six. the van brings the orders. "
        "the clerk packs the orders. the van leaves at noon. ") * 4

In [ ]:
import numpy as np

text = ("the shop opens at nine. the shop closes at six. the van brings the orders. "
        "the clerk packs the orders. the van leaves at noon. ") * 4
chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: "".join(itos[i] for i in ids)
V = len(chars); data = np.array(encode(text))

def softmax(z):
    z = z - z.max(-1, keepdims=True); e = np.exp(z); return e / e.sum(-1, keepdims=True)
def layernorm(x, eps=1e-5):
    return (x - x.mean(-1, keepdims=True)) / np.sqrt(x.var(-1, keepdims=True) + eps)
def gelu(x): return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

def init(rng, V, T, d=16, heads=2, layers=2):
    s = 0.02
    P = {"tok": rng.normal(0, s, (V, d)), "pos": rng.normal(0, s, (T, d)), "heads": heads, "layers": []}
    for _ in range(layers):
        P["layers"].append({"Wqkv": rng.normal(0, s, (d, 3*d)), "Wo": rng.normal(0, s, (d, d)),
                            "W1": rng.normal(0, s, (d, 4*d)), "W2": rng.normal(0, s, (4*d, d))})
    return P

def attention(x, L, heads):
    T, d = x.shape; hd = d // heads
    q, k, v = np.split(x @ L["Wqkv"], 3, axis=-1)
    q, k, v = [m.reshape(T, heads, hd).transpose(1, 0, 2) for m in (q, k, v)]
    scores = q @ k.transpose(0, 2, 1) / np.sqrt(hd)
    scores = np.where(np.tril(np.ones((T, T), bool)), scores, -1e9)   # causal mask
    w = softmax(scores)
    return (w @ v).transpose(1, 0, 2).reshape(T, d) @ L["Wo"], w

def forward(P, ids):
    T = len(ids); x = P["tok"][ids] + P["pos"][:T]
    for L in P["layers"]:
        a, w = attention(layernorm(x), L, P["heads"]); x = x + a
        x = x + gelu(layernorm(x) @ L["W1"]) @ L["W2"]
    return layernorm(x) @ P["tok"].T, w          # weight tying

def loss_fn(logits, targets):
    p = softmax(logits); return -np.log(p[np.arange(len(targets)), targets]).mean()

In [ ]:
print("vocab size V =", V, " chars:", repr("".join(chars)))
ids = encode("the shop"); print("encode:", ids, "decode:", decode(ids))
T = 16; P = init(np.random.default_rng(0), V, T)
n_params = P["tok"].size + P["pos"].size + sum(m.size for L in P["layers"] for m in L.values())
print("parameters:", n_params)
x, y = data[:T], data[1:T+1]
logits, w = forward(P, x)
print("logits shape:", logits.shape, " loss:", round(loss_fn(logits, y), 4), " ln(V):", round(np.log(V), 4))
print("attention row sums:", np.round(w[0].sum(-1)[:4], 3), " weights above diagonal max:", w[0][np.triu_indices(T, 1)].max())
x2 = x.copy(); x2[10] = (x2[10] + 1) % V
l2, _ = forward(P, x2)
print("rows 0-9 unchanged after editing token 10:", np.allclose(logits[:10], l2[:10]), "| row 10 changed:", not np.allclose(logits[10], l2[10]))
counts = np.ones((V, V))
for a, b in zip(data[:-1], data[1:]): counts[a, b] += 1
probs = counts / counts.sum(1, keepdims=True)
nll = -np.mean([np.log(probs[a, b]) for a, b in zip(data[:-1], data[1:])])
print("bigram baseline loss:", round(nll, 4), " perplexity:", round(np.exp(nll), 2))

In [ ]:
# model = MiniGPT(vocab_size, block_size=64, d=64, heads=4, layers=4)   # same structure as above
# opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
# for step in range(5000):
#     ix = torch.randint(len(train_ids) - block_size - 1, (batch,))
#     x = torch.stack([train_ids[i:i+block_size] for i in ix])
#     y = torch.stack([train_ids[i+1:i+block_size+1] for i in ix])      # shifted by one
#     logits = model(x); loss = F.cross_entropy(logits.view(-1, V), y.view(-1))
#     opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
#     # every 200 steps: evaluate loss on held-out val_ids; generate a sample

## 16.3 Project: A Sequence Model for Next-Character Prediction (NEW)

In [ ]:
import numpy as np

text = ("the shop opens at nine. the shop closes at six. the van brings the orders. "
        "the clerk packs the orders. the van leaves at noon. ") * 4
chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
V = len(chars); data = np.array([stoi[c] for c in text])

def init(rng, V, H):
    return {"Wxh": rng.normal(0, 0.1, (V, H)), "Whh": rng.normal(0, 0.1, (H, H)), "bh": np.zeros(H),
            "Why": rng.normal(0, 0.1, (H, V)), "by": np.zeros(V)}

def softmax(z): e = np.exp(z - z.max()); return e / e.sum()

def loss_and_grads(P, xs, ys, h0):
    hs = {-1: h0}; ps = {}; loss = 0
    for t, (x, y) in enumerate(zip(xs, ys)):                  # forward through time
        hs[t] = np.tanh(P["Wxh"][x] + hs[t-1] @ P["Whh"] + P["bh"])
        ps[t] = softmax(hs[t] @ P["Why"] + P["by"]); loss -= np.log(ps[t][y])
    loss /= len(xs)
    g = {k: np.zeros_like(v) for k, v in P.items()}; dh_next = np.zeros_like(h0)
    for t in reversed(range(len(xs))):                        # backprop through time
        dy = ps[t].copy(); dy[ys[t]] -= 1; dy /= len(xs)
        g["Why"] += np.outer(hs[t], dy); g["by"] += dy
        dh = dy @ P["Why"].T + dh_next; dz = dh * (1 - hs[t]**2)
        g["Wxh"][xs[t]] += dz; g["Whh"] += np.outer(hs[t-1], dz); g["bh"] += dz
        dh_next = dz @ P["Whh"].T
    for k in g: np.clip(g[k], -5, 5, out=g[k])               # gradient clipping
    return loss, g, hs[len(xs)-1]

def sample(P, seed_char, n, H, rng, temp=1.0):
    h = np.zeros(H); x = stoi[seed_char]; out = [seed_char]
    for _ in range(n):
        h = np.tanh(P["Wxh"][x] + h @ P["Whh"] + P["bh"])
        p = softmax((h @ P["Why"] + P["by"]) / temp); x = rng.choice(V, p=p); out.append(itos[x])
    return "".join(out)

In [ ]:
H, T = 32, 20; rng = np.random.default_rng(0); P = init(rng, V, H)
xs, ys = data[:T], data[1:T+1]
loss, g, _ = loss_and_grads(P, xs, ys, np.zeros(H))
worst = 0
for name in P:
    flat = P[name].ravel()
    for i in np.random.default_rng(1).choice(flat.size, 5, replace=False):
        old = flat[i]; eps = 1e-5
        flat[i] = old + eps; lp = loss_and_grads(P, xs, ys, np.zeros(H))[0]
        flat[i] = old - eps; lm = loss_and_grads(P, xs, ys, np.zeros(H))[0]; flat[i] = old
        num = (lp - lm) / (2*eps); ana = g[name].ravel()[i]
        worst = max(worst, abs(num - ana) / max(1e-12, abs(num) + abs(ana)))
print(f"V={V}  initial loss {loss:.4f}  ln(V)={np.log(V):.4f}  gradcheck worst rel error {worst:.2e}")
mem = {k: np.zeros_like(v) for k, v in P.items()}; lr = 0.1; pos = 0; h = np.zeros(H)
for it in range(1, 3001):
    if pos + T + 1 >= len(data): pos = 0; h = np.zeros(H)
    xs, ys = data[pos:pos+T], data[pos+1:pos+T+1]; pos += T
    loss, g, h = loss_and_grads(P, xs, ys, h)
    for k in P: mem[k] += g[k]**2; P[k] -= lr * g[k] / np.sqrt(mem[k] + 1e-8)   # Adagrad
    if it in (1, 100, 500, 1000, 3000): print(f"iter {it:4d} loss {loss:.4f}")
print("sample:", repr(sample(P, "t", 80, H, np.random.default_rng(3))))
print("temp 0.3:", repr(sample(P, "t", 80, H, np.random.default_rng(3), 0.3)))

## 16.4 Checkpoints, Expected Numbers and Reading a Detector Honestly

The three projects in this chapter are easy to start and hard to judge. This section gives you numbers to check against and a worked example of why one mAP figure can mislead.

In [ ]:
import math
V = 65                        # characters in a tiny-Shakespeare style vocabulary
print("uniform loss", round(math.log(V), 3), " uniform perplexity", V)
for loss in (4.17, 2.5, 1.6, 1.3):
    print(f"loss {loss}  perplexity {math.exp(loss):.1f}  bits/char {loss/math.log(2):.2f}")

uniform loss 4.174  uniform perplexity 65
loss 4.17  perplexity 64.7  bits/char 6.02
loss 2.5  perplexity 12.2  bits/char 3.61
loss 1.6  perplexity 5.0  bits/char 2.31
loss 1.3  perplexity 3.7  bits/char 1.88


In [ ]:
import numpy as np
# six detections of one class, sorted by confidence; iou = overlap with its best ground-truth box
# (each ground-truth box can be matched once). 5 ground-truth boxes exist in total.
score = np.array([0.95, 0.90, 0.80, 0.70, 0.60, 0.50])
iou   = np.array([0.92, 0.81, 0.55, 0.88, 0.30, 0.72])
n_gt = 5

def ap_at(th):
    tp = (iou >= th).astype(float)
    cum_tp = np.cumsum(tp); cum_fp = np.cumsum(1 - tp)
    rec = cum_tp / n_gt; prec = cum_tp / (cum_tp + cum_fp)
    # all-point interpolation: precision envelope, summed over recall steps
    env = np.maximum.accumulate(prec[::-1])[::-1]
    r = np.concatenate([[0], rec]); e = np.concatenate([[env[0]], env])
    return float(((r[1:] - r[:-1]) * e[1:]).sum()), prec, rec

for th in (0.5, 0.75, 0.9):
    ap, prec, rec = ap_at(th)
    print(f"IoU>={th}: precision {np.round(prec,2)} recall {np.round(rec,2)} AP {ap:.3f}")
print("mean AP over IoU 0.5, 0.75, 0.9:", round(np.mean([ap_at(t)[0] for t in (0.5, 0.75, 0.9)]), 3))

IoU>=0.5: precision [1.   1.   1.   1.   0.8  0.83] recall [0.2 0.4 0.6 0.8 0.8 1. ] AP 0.967
IoU>=0.75: precision [1.   1.   0.67 0.75 0.6  0.5 ] recall [0.2 0.4 0.4 0.6 0.6 0.6] AP 0.550
IoU>=0.9: precision [1.   0.5  0.33 0.25 0.2  0.17] recall [0.2 0.2 0.2 0.2 0.2 0.2] AP 0.200
mean AP over IoU 0.5, 0.75, 0.9: 0.572


## Chapter recap

## Practice set

**Task 1 (easy).** Convert the COCO box `[10, 20, 30, 40]` to xyxy. **Answer:** `[10, 20, 40, 60]`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch16.html